# Mini Projet : Assistant de Sentiment avec Fine-Tuning BERT

Ce notebook guide un débutant à travers le processus de réglage fin (fine-tuning) d'un modèle BERT pour l'analyse de sentiment.

### Étapes du Workflow :
1. **Installation & Configuration** : Préparation de l'environnement.
2. **Chargement des données** : Utilisation du dataset IMDB.
3. **Pipeline de données** : Tokenisation spécifique à BERT.
4. **Configuration du Modèle** : Chargement de BERT pré-entraîné.
5. **Entraînement** : Fine-tuning sur les critiques de films.
6. **Évaluation & Inférence** : Test sur de nouvelles phrases.

In [ ]:
# Étape 1 : Installation des bibliothèques nécessaires
# transformers : pour BERT
# datasets/tensorflow_datasets : pour les données
# accelerate : pour optimiser l'entraînement sur GPU
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

In [ ]:
import os
import sys



# 2. Configure environment for Keras 3 compatibility
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# 3. Import logic with direct path fallback
try:
    from transformers import BertTokenizer, TFBertForSequenceClassification
except (ImportError, AttributeError):
    from transformers import BertTokenizer
    try:
        from transformers.models.bert.modeling_tf_bert import TFBertForSequenceClassification
    except ImportError:
        # Some versions require this specific internal path
        from transformers import TFBertModel as TFBertForSequenceClassification

# Verification
print("Version TensorFlow   :", tf.__version__)
import transformers
print("Version Transformers :", transformers.__version__)
print("GPU détectés         :", tf.config.list_physical_devices('GPU'))

### Étape 2 : Chargement du Dataset IMDB
Nous utilisons 25 000 avis pour l'entraînement et 25 000 pour le test.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)

# Aperçu d'un échantillon
for text, label in ds_train.take(1):
    print("Sentiment :", "Positif" if label.numpy() else "Négatif")
    print("Extrait :", text.numpy().decode()[:200], "...")

### Étape 3 : Tokenisation et Pipeline
BERT ne lit pas directement du texte, il a besoin de 'Token IDs' (nombres).

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

# Chargement du tokenizer BERT non-accentué (uncased)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

def encode_review(review_input):
    # Gestion des différents types d'entrée (bytes ou tenseurs)
    if isinstance(review_input, bytes): review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"): review_text = review_input.numpy().decode("utf-8")
    else: review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True, # Ajoute [CLS] et [SEP]
        max_length=MAX_LENGTH,
        padding="max_length",    # Remplit avec des 0 pour avoir la même taille
        truncation=True,        # Coupe si c'est trop long
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    # Utilisation de py_function pour intégrer le tokenizer Python dans le graphe TensorFlow
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {"input_ids": encoded[0], "attention_mask": encoded[1], "token_type_ids": encoded[2]}, label

# Préparation finale des datasets avec mise en cache et préchargement (prefetch)
train_ds = ds_train.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE).shuffle(2000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = ds_test.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

### Étape 4 : Initialisation du Modèle

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2 # Binaire : Positif ou Négatif
)

# Configuration de l'optimiseur avec un taux d'apprentissage très faible (standard pour BERT)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

### Étape 5 : Entraînement
Nous entraînons sur 2 époques (ce qui suffit souvent avec BERT pour obtenir >90% de précision).

In [ ]:
EPOCHS = 2
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

### Étape 6 : Inférence et Analyse
Créons une fonction pour tester des phrases personnalisées.

In [ ]:
def predict_sentiment(text: str):
    # Préparation du texte
    inputs = encode_review(text)
    # Conversion en tenseurs TensorFlow et ajout d'une dimension de batch
    input_ids = tf.expand_dims(inputs['input_ids'], 0)
    mask = tf.expand_dims(inputs['attention_mask'], 0)

    # Prédiction
    outputs = model(input_ids, attention_mask=mask)
    logits = outputs.logits
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]

    label = "Positif" if np.argmax(probs) == 1 else "Négatif"
    return label, float(np.max(probs))

# Test
custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Phrase : {custom_sentence}")
print(f"Prédiction : {label} (Confiance : {confidence:.3f})")

### Questions de Réflexion

1. **Quel levier a le plus amélioré les résultats ?**
   *Réponse :* Généralement, c'est le pré-entraînement de BERT lui-même (transfer learning). Sinon, augmenter le `MAX_LENGTH` permet de capturer plus de contexte dans les longs avis.

2. **Où ajouteriez-vous des garde-fous avant le déploiement ?**
   *Réponse :* Sur le score de confiance (`confidence`). Si la confiance est inférieure à 70%, il vaut mieux envoyer le ticket à un humain plutôt que d'automatiser une réponse erronée.

3. **Quels intervenants en bénéficient le plus ?**
   *Réponse :* L'équipe Support (priorisation), les Product Managers (analyse des points de friction) et les Data Analysts (tendances de satisfaction).